# ZestXML: PyTorch port vs the C++ reference

Runs both implementations on **GZ-Eurlex-4.3K** (45k train / 6k test points, 4.3k labels,
100k point features) with identical hyper-parameters, and reports P@k / nDCG@k / PSP@k,
wall time and peak GPU memory.

Two things this is meant to settle, neither of which could be tested where the port was written:
1. whether the GPU path is actually fast — on CPU the C++ is 2-4x quicker;
2. whether it holds up at a real label scale (previous runs topped out at 3.2k labels).

**Runtime → Change runtime type → A100 GPU** before running.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf zestxml
!git clone -q -b claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml.git
%cd /content/zestxml
!ls

## 1. Sanity check (~1 min)
Unit tests plus the parity harness, which builds the C++ binary and checks the mined pattern
and scores against it. If this fails, stop here — the benchmark below will not mean anything.

In [ ]:
!pip install -q pytest
!python -m pytest tests -q

## 2. The benchmark
Downloads the dataset (~1 GB from Drive), builds `./run`, then trains and predicts with both
implementations. Expect roughly 20-40 min in total, most of it the C++ baseline.

Pass `torch` or `cpp` as a second argument to run only one side.

In [ ]:
!bash tools/colab_benchmark.sh GZ-Eurlex-4.3K both

## 3. Cross-check the metrics with pyxclib
`tools/eval_xc.py` implements P@k / nDCG@k / PSP@k directly so it can run without pyxclib.
This cell runs the repo's original `metrics.py` on the same score matrix, so the two
implementations of the *metrics* can be compared as well. (`np.bool` is patched in because
`metrics.py` predates numpy 1.24.)

In [ ]:
!pip install -q pandas tabulate tqdm scikit-learn
!pip install -q git+https://github.com/kunaldahiya/pyxclib.git
!mkdir -p Results/GZ-Eurlex-4.3K && cp Results/GZ-Eurlex-4.3K-torch/score_mat.bin Results/GZ-Eurlex-4.3K/

import sys, numpy as np
np.bool = bool
sys.argv = ['metrics.py', 'GZ-Eurlex-4.3K']
exec(open('metrics.py').read())

## 4. If it runs out of GPU memory
Every stage is chunked, so this is a knob, not a wall. Lower these and rerun the torch side
(`bash tools/colab_benchmark.sh GZ-Eurlex-4.3K torch`), or edit them in the script:

| flag | meaning | try |
| --- | --- | --- |
| `-max_elems` | non-zeros expanded per batch | `16777216` |
| `-dense_elems` | entries in a dense working block | `8388608` |
| `-batch_size` | points per gradient step | `128` |

Please send back the full output of cell 2 — the `[STAT]` lines carry the timings, peak GPU
memory, shortlist recall and the metric tables for both implementations.

In [ ]:
# rerun just the PyTorch side with smaller working blocks
# !bash tools/colab_benchmark.sh GZ-Eurlex-4.3K torch